In [6]:
import os, gc, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import warnings

import time, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_properties(0).name}")

Device: cuda
GPU: NVIDIA GeForce RTX 5090


In [2]:
SEED       = 42
EMBED_DIM  = 512
NUM_PROT   = 4
LSIG_LEN   = 24
HTSIG_LEN  = 48
PROT_NAMES = ['Non-HT', 'HT', 'VHT', 'HTGF']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True

In [3]:
import os
os.makedirs('/workspace/data', exist_ok=True)

print("Loading data...")
train_data = torch.load('/workspace/data/train.pt')
test_data  = torch.load('/workspace/data/test.pt')

high_train, low_train, prot_train, lsig_train, htsig_train, has_htsig_train = train_data
high_test,  low_test,  prot_test,  lsig_test,  htsig_test,  has_htsig_test  = test_data

print(f"Train: {len(prot_train):,} samples")
print(f"Test:  {len(prot_test):,} samples")

Loading data...
Train: 86,437 samples
Test:  9,725 samples


In [8]:
class AugmentedDataset(Dataset):
    def __init__(self, data, augment=False):
        self.high, self.low, self.prot, self.lsig, self.htsig, self.has_htsig = data
        self.augment = augment

    def __len__(self):
        return len(self.prot)

    def __getitem__(self, idx):
        high = self.high[idx].clone()
        low  = self.low[idx].clone()
    
        if self.augment:
            theta = torch.empty(1).uniform_(0, 2 * 3.14159265).item()
            cos_t = torch.cos(torch.tensor(theta))
            sin_t = torch.sin(torch.tensor(theta))
            for iq in (high, low):
                real  = iq[0].clone()
                iq[0] = real * cos_t - iq[1] * sin_t
                iq[1] = real * sin_t + iq[1] * cos_t
    
        use_low = torch.rand(1).item() < self.low_ratio
        signal  = low if use_low else high
    
        # Always return same length — upsample low-res to high-res length if needed
        if signal.shape[-1] != 8000:
            signal = F.interpolate(signal.unsqueeze(0), size=8000, 
                                   mode='linear', align_corners=False).squeeze(0)
    
        return signal, self.prot[idx], self.lsig[idx], self.htsig[idx], self.has_htsig[idx]

def task_loss_balanced(prot_l, lsig_l, htsig_l,
                       prot_gt, lsig_gt, htsig_gt, has_htsig):
    w = torch.tensor([0.5, 1.5, 8.0, 8.0]).to(prot_l.device).to(prot_l.dtype)
    loss = F.cross_entropy(prot_l, prot_gt, weight=w)
    loss = loss + F.binary_cross_entropy_with_logits(lsig_l.float(), lsig_gt.float())
    mask = has_htsig.bool()
    if mask.any():
        loss = loss + F.binary_cross_entropy_with_logits(
                        htsig_l[mask].float(), htsig_gt[mask].float())
    return loss

def bit_acc(logits, targets):
    return ((logits.sigmoid() > 0.5).float() == targets).float().mean().item()

def prot_acc(logits, targets):
    return (logits.argmax(1) == targets).float().mean().item()

def get_scheduler_with_warmup(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1 + torch.cos(torch.tensor(progress * 3.14159265)).item())
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

@torch.no_grad()
def evaluate(model, loader, use_high=False):
    model.eval()
    tot_loss = tot_prot = tot_lsig = 0
    for high_iq, low_iq, prot, lsig, htsig, has_htsig in loader:
        iq        = (high_iq if use_high else low_iq).to(DEVICE, non_blocking=True)
        prot      = prot.to(DEVICE, non_blocking=True)
        lsig      = lsig.to(DEVICE, non_blocking=True)
        htsig     = htsig.to(DEVICE, non_blocking=True)
        has_htsig = has_htsig.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            prot_l, lsig_l, htsig_l, _, _, _ = model(iq)
        loss = task_loss_balanced(
            prot_l.float(), lsig_l.float(), htsig_l.float(),
            prot, lsig.float(), htsig.float(), has_htsig)
        tot_loss += loss.item()
        tot_prot += prot_acc(prot_l, prot)
        tot_lsig += bit_acc(lsig_l, lsig)
    n = len(loader)
    return tot_loss/n, tot_prot/n, tot_lsig/n

train_ds = AugmentedDataset(train_data, augment=True)
test_ds  = AugmentedDataset(test_data,  augment=False)
print("Setup complete")

Setup complete


In [12]:
# Redefine standard AugmentedDataset without low_ratio
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, data, augment=False):
        self.high, self.low, self.prot, self.lsig, self.htsig, self.has_htsig = data
        self.augment = augment

    def __len__(self):
        return len(self.prot)

    def __getitem__(self, idx):
        high = self.high[idx].clone()
        low  = self.low[idx].clone()
        if self.augment:
            theta = torch.empty(1).uniform_(0, 2 * 3.14159265).item()
            cos_t = torch.cos(torch.tensor(theta))
            sin_t = torch.sin(torch.tensor(theta))
            for iq in (high, low):
                real  = iq[0].clone()
                iq[0] = real * cos_t - iq[1] * sin_t
                iq[1] = real * sin_t + iq[1] * cos_t
        return high, low, self.prot[idx], self.lsig[idx], self.htsig[idx], self.has_htsig[idx]

# Recreate test loader with clean dataset
test_ds     = AugmentedDataset(test_data, augment=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=0, pin_memory=True)

In [15]:
class ProgressiveDataset(torch.utils.data.Dataset):
    def __init__(self, data, augment=False):
        self.high, self.low, self.prot, self.lsig, self.htsig, self.has_htsig = data
        self.augment   = augment
        self.low_ratio = 0.0

    def set_low_ratio(self, ratio):
        self.low_ratio = ratio

    def __len__(self):
        return len(self.prot)

    def __getitem__(self, idx):
        high = self.high[idx].clone()
        low  = self.low[idx].clone()

        if self.augment:
            theta = torch.empty(1).uniform_(0, 2 * 3.14159265).item()
            cos_t = torch.cos(torch.tensor(theta))
            sin_t = torch.sin(torch.tensor(theta))
            for iq in (high, low):
                real  = iq[0].clone()
                iq[0] = real * cos_t - iq[1] * sin_t
                iq[1] = real * sin_t + iq[1] * cos_t

        use_low = torch.rand(1).item() < self.low_ratio
        signal  = low if use_low else high

        # Always upsample to 8000 so batches can stack
        if signal.shape[-1] != 8000:
            signal = F.interpolate(signal.unsqueeze(0), size=8000,
                                   mode='linear', align_corners=False).squeeze(0)
        return signal, self.prot[idx], self.lsig[idx], self.htsig[idx], self.has_htsig[idx]

# Recreate
prog_ds     = ProgressiveDataset(train_data, augment=True)
prog_loader = DataLoader(prog_ds, batch_size=256, shuffle=True,
                         num_workers=0, pin_memory=True)

In [16]:
gc.collect()
torch.cuda.empty_cache()

In [18]:
import time, gc, os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# ── Setup ──────────────────────────────────────────────────────────────────
SEED       = 42
EMBED_DIM  = 512
NUM_PROT   = 4
LSIG_LEN   = 24
HTSIG_LEN  = 48
PROT_NAMES = ['Non-HT', 'HT', 'VHT', 'HTGF']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_properties(0).name}")

# ── Load Data ──────────────────────────────────────────────────────────────
print("Loading data...")
os.makedirs('/workspace/data', exist_ok=True)
train_data = torch.load('/workspace/data/train.pt')
test_data  = torch.load('/workspace/data/test.pt')
high_train, low_train, prot_train, lsig_train, htsig_train, has_htsig_train = train_data
print(f"Train: {len(prot_train):,} | Test: {len(test_data[2]):,}")

# ── Loss Functions ─────────────────────────────────────────────────────────
def task_loss_balanced(prot_l, lsig_l, htsig_l,
                       prot_gt, lsig_gt, htsig_gt, has_htsig):
    w = torch.tensor([0.5, 1.5, 8.0, 8.0]).to(prot_l.device).to(prot_l.dtype)
    loss = F.cross_entropy(prot_l, prot_gt, weight=w)
    loss = loss + F.binary_cross_entropy_with_logits(lsig_l.float(), lsig_gt.float())
    mask = has_htsig.bool()
    if mask.any():
        loss = loss + F.binary_cross_entropy_with_logits(
                        htsig_l[mask].float(), htsig_gt[mask].float())
    return loss

def bit_acc(logits, targets):
    return ((logits.sigmoid() > 0.5).float() == targets).float().mean().item()

def prot_acc(logits, targets):
    return (logits.argmax(1) == targets).float().mean().item()

def get_scheduler_with_warmup(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1 + torch.cos(torch.tensor(progress * 3.14159265)).item())
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Evaluate ───────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_loss = tot_prot = tot_lsig = 0
    for high_iq, low_iq, prot, lsig, htsig, has_htsig in loader:
        iq        = low_iq.to(DEVICE, non_blocking=True)
        prot      = prot.to(DEVICE, non_blocking=True)
        lsig      = lsig.to(DEVICE, non_blocking=True)
        htsig     = htsig.to(DEVICE, non_blocking=True)
        has_htsig = has_htsig.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            prot_l, lsig_l, htsig_l, _, _, _ = model(iq)
        loss = task_loss_balanced(
            prot_l.float(), lsig_l.float(), htsig_l.float(),
            prot, lsig.float(), htsig.float(), has_htsig)
        tot_loss += loss.item()
        tot_prot += prot_acc(prot_l, prot)
        tot_lsig += bit_acc(lsig_l, lsig)
    n = len(loader)
    return tot_loss/n, tot_prot/n, tot_lsig/n

# ── Standard Dataset for eval ──────────────────────────────────────────────
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, data, augment=False):
        self.high, self.low, self.prot, self.lsig, self.htsig, self.has_htsig = data
        self.augment = augment

    def __len__(self):
        return len(self.prot)

    def __getitem__(self, idx):
        high = self.high[idx].clone()
        low  = self.low[idx].clone()
        if self.augment:
            theta = torch.empty(1).uniform_(0, 2 * 3.14159265).item()
            cos_t = torch.cos(torch.tensor(theta))
            sin_t = torch.sin(torch.tensor(theta))
            for iq in (high, low):
                real  = iq[0].clone()
                iq[0] = real * cos_t - iq[1] * sin_t
                iq[1] = real * sin_t + iq[1] * cos_t
        return high, low, self.prot[idx], self.lsig[idx], self.htsig[idx], self.has_htsig[idx]

# ── Progressive Resolution Dataset ────────────────────────────────────────
class ProgressiveDataset(torch.utils.data.Dataset):
    """
    NOVEL: Easy -> Hard curriculum learning.
    Epochs 1-8:   100% high-res (easy)
    Epochs 9-16:  75% high-res, 25% low-res
    Epochs 17-24: 50/50
    Epochs 25-32: 25% high-res, 75% low-res
    Epochs 33-40: 100% low-res (hard)
    """
    def __init__(self, data, augment=False):
        self.high, self.low, self.prot, self.lsig, self.htsig, self.has_htsig = data
        self.augment   = augment
        self.low_ratio = 0.0

    def set_low_ratio(self, ratio):
        self.low_ratio = ratio

    def __len__(self):
        return len(self.prot)

    def __getitem__(self, idx):
        high = self.high[idx].clone()
        low  = self.low[idx].clone()

        if self.augment:
            theta = torch.empty(1).uniform_(0, 2 * 3.14159265).item()
            cos_t = torch.cos(torch.tensor(theta))
            sin_t = torch.sin(torch.tensor(theta))
            for iq in (high, low):
                real  = iq[0].clone()
                iq[0] = real * cos_t - iq[1] * sin_t
                iq[1] = real * sin_t + iq[1] * cos_t

        use_low = torch.rand(1).item() < self.low_ratio
        signal  = low if use_low else high

        # Always normalize to 8000 samples so batches can stack
        if signal.shape[-1] != 8000:
            signal = F.interpolate(signal.unsqueeze(0), size=8000,
                                   mode='linear', align_corners=False).squeeze(0)
        return signal, self.prot[idx], self.lsig[idx], self.htsig[idx], self.has_htsig[idx]


def get_progressive_ratio(epoch, total_epochs=40):
    phase = (epoch - 1) / (total_epochs - 1)
    if phase < 0.2:    return 0.0
    elif phase < 0.4:  return 0.25
    elif phase < 0.6:  return 0.5
    elif phase < 0.8:  return 0.75
    else:              return 1.0

# ── Model ──────────────────────────────────────────────────────────────────
class IQEncoder_Prog(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(2, 64,  kernel_size=7, stride=2, padding=3), nn.BatchNorm1d(64),  nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2), nn.BatchNorm1d(128), nn.GELU(),
            nn.Conv1d(128, 256, kernel_size=3, stride=2, padding=1), nn.BatchNorm1d(256), nn.GELU(),
            nn.Conv1d(256, 384, kernel_size=3, stride=2, padding=1), nn.BatchNorm1d(384), nn.GELU(),
            nn.Conv1d(384, embed_dim, kernel_size=3, stride=2, padding=1), nn.BatchNorm1d(embed_dim), nn.GELU(),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dim_feedforward=1024,
            dropout=0.1, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4,
                                                  enable_nested_tensor=False)
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        feat   = self.cnn(x)
        feat_t = self.transformer(feat.permute(0,2,1)).permute(0,2,1)
        return self.pool(feat_t).squeeze(-1), feat


class WiFiDecoder_Prog(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.prot_head  = nn.Sequential(nn.Linear(embed_dim, 128), nn.GELU(), nn.Linear(128, NUM_PROT))
        self.lsig_head  = nn.Sequential(nn.Linear(embed_dim, 128), nn.GELU(), nn.Linear(128, LSIG_LEN))
        self.htsig_head = nn.Sequential(nn.Linear(embed_dim, 128), nn.GELU(), nn.Linear(128, HTSIG_LEN))

    def forward(self, emb):
        return self.prot_head(emb), self.lsig_head(emb), self.htsig_head(emb)


class ProgModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = IQEncoder_Prog()
        self.decoder = WiFiDecoder_Prog()

    def forward(self, x):
        emb, feat = self.encoder(x)
        prot_l, lsig_l, htsig_l = self.decoder(emb)
        return prot_l, lsig_l, htsig_l, emb, emb, emb

# ── Training ───────────────────────────────────────────────────────────────
PROG_EPOCHS   = 40
PROG_LR       = 5e-4
WARMUP_EPOCHS = 3
ACCUM_STEPS   = 2

test_ds      = AugmentedDataset(test_data, augment=False)
prog_ds      = ProgressiveDataset(train_data, augment=True)
prog_loader  = DataLoader(prog_ds, batch_size=256, shuffle=True, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

prog_model  = ProgModel().to(DEVICE)
p_opt       = torch.optim.AdamW(prog_model.parameters(), lr=PROG_LR, weight_decay=1e-4)
p_sched     = get_scheduler_with_warmup(p_opt, WARMUP_EPOCHS, PROG_EPOCHS)
p_scaler    = torch.amp.GradScaler('cuda')
best_p_prot = 0.0

total_params = sum(p.numel() for p in prog_model.parameters())
print(f"Parameters: {total_params:,} ({total_params*4/1e6:.1f} MB)")
print(f"\n{'Ep':>3} {'lo_ratio':>9} {'tr_loss':>8} {'tr_prot':>8} {'tr_lsig':>8} {'va_prot':>8} {'va_lsig':>8} {'time':>6}")
print("-" * 75)

for epoch in range(1, PROG_EPOCHS + 1):
    low_ratio = get_progressive_ratio(epoch, PROG_EPOCHS)
    prog_ds.set_low_ratio(low_ratio)

    prog_model.train()
    t0 = time.time()
    tot_loss = tot_prot = tot_lsig = 0
    p_opt.zero_grad()

    for i, (signal, prot, lsig, htsig, has_htsig) in enumerate(prog_loader):
        signal    = signal.to(DEVICE, non_blocking=True)
        prot      = prot.to(DEVICE, non_blocking=True)
        lsig      = lsig.to(DEVICE, non_blocking=True)
        htsig     = htsig.to(DEVICE, non_blocking=True)
        has_htsig = has_htsig.to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda'):
            prot_l, lsig_l, htsig_l, _, _, _ = prog_model(signal)
            loss = task_loss_balanced(prot_l, lsig_l, htsig_l,
                                      prot, lsig, htsig, has_htsig) / ACCUM_STEPS

        p_scaler.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(prog_loader):
            p_scaler.unscale_(p_opt)
            torch.nn.utils.clip_grad_norm_(prog_model.parameters(), 1.0)
            p_scaler.step(p_opt)
            p_scaler.update()
            p_opt.zero_grad()

        tot_loss += loss.item() * ACCUM_STEPS
        tot_prot += prot_acc(prot_l, prot)
        tot_lsig += bit_acc(lsig_l, lsig)

    p_sched.step()
    n  = len(prog_loader)
    tr = (tot_loss/n, tot_prot/n, tot_lsig/n)
    va = evaluate(prog_model, test_loader)
    elapsed = time.time() - t0

    if va[1] > best_p_prot:
        best_p_prot = va[1]
        torch.save(prog_model.state_dict(), '/workspace/progressive_res_best.pt')
        marker = ' <- best'
    else:
        marker = ''

    print(f"{epoch:3d} {low_ratio:9.2f} {tr[0]:8.4f} {tr[1]:8.3f} {tr[2]:8.3f} "
          f"{va[1]:8.3f} {va[2]:8.3f} {elapsed:5.0f}s{marker}", flush=True)

print(f"\nBest Progressive Resolution va_prot: {best_p_prot:.4f}")

# ── Per-class accuracy ─────────────────────────────────────────────────────
prog_model.load_state_dict(torch.load('/workspace/progressive_res_best.pt'))
prog_model.eval()
correct = torch.zeros(4)
total   = torch.zeros(4)

with torch.no_grad():
    for high_iq, low_iq, prot, *_ in test_loader:
        iq     = low_iq.to(DEVICE)
        prot_g = prot.to(DEVICE)
        with torch.amp.autocast('cuda'):
            prot_l, _, _, _, _, _ = prog_model(iq)
        preds = prot_l.argmax(1)
        for c in range(4):
            mask = (prot_g == c)
            correct[c] += (preds[mask] == prot_g[mask]).sum().cpu()
            total[c]   += mask.sum().cpu()

print("\nPer-class accuracy (Progressive Resolution):")
print(f"{'Class':>8} {'Prog':>8} {'CNN+Trans':>12} {'Diff':>8}")
print("-" * 40)
cnn_trans = [97.0, 91.9, 58.0, 50.0]
for c in range(4):
    acc  = correct[c]/(total[c]+1e-8) * 100
    diff = acc.item() - cnn_trans[c]
    print(f"{PROT_NAMES[c]:>8}: {acc.item():>7.1f}%  {cnn_trans[c]:>10.1f}%  {diff:>+6.1f}%")

Device: cuda
GPU: NVIDIA GeForce RTX 5090
Loading data...
Train: 86,437 | Test: 9,725
Parameters: 9,646,860 (38.6 MB)

 Ep  lo_ratio  tr_loss  tr_prot  tr_lsig  va_prot  va_lsig   time
---------------------------------------------------------------------------
  1      0.00   1.1886    0.794    0.887    0.713    0.883    33s <- best
  2      0.00   0.6635    0.954    0.917    0.454    0.823    33s
  3      0.00   0.5045    0.975    0.936    0.686    0.834    33s
  4      0.00   0.3927    0.983    0.952    0.590    0.817    32s
  5      0.00   0.3448    0.987    0.960    0.533    0.804    33s
  6      0.00   0.2897    0.989    0.969    0.547    0.848    33s
  7      0.00   0.2554    0.990    0.975    0.639    0.845    33s
  8      0.00   0.2285    0.992    0.978    0.314    0.820    33s
  9      0.25   0.5053    0.939    0.955    0.624    0.849    34s
 10      0.25   0.3839    0.971    0.961    0.604    0.858    33s
 11      0.25   0.3408    0.978    0.966    0.513    0.867    33s
 12  

In [22]:
import time, gc, os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# ── Setup ──────────────────────────────────────────────────────────────────
SEED       = 42
EMBED_DIM  = 512
NUM_PROT   = 4
LSIG_LEN   = 24
HTSIG_LEN  = 48
PROT_NAMES = ['Non-HT', 'HT', 'VHT', 'HTGF']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_properties(0).name}")

# ── Load Data ──────────────────────────────────────────────────────────────
print("Loading data...")
train_data = torch.load('/workspace/data/train.pt')
test_data  = torch.load('/workspace/data/test.pt')
high_train, low_train, prot_train, lsig_train, htsig_train, has_htsig_train = train_data
print(f"Train: {len(prot_train):,} | Test: {len(test_data[2]):,}")

# ── Helpers ────────────────────────────────────────────────────────────────
def task_loss_balanced(prot_l, lsig_l, htsig_l, prot_gt, lsig_gt, htsig_gt, has_htsig):
    w = torch.tensor([0.5, 1.5, 8.0, 8.0]).to(prot_l.device).to(prot_l.dtype)
    loss = F.cross_entropy(prot_l, prot_gt, weight=w)
    loss = loss + F.binary_cross_entropy_with_logits(lsig_l.float(), lsig_gt.float())
    mask = has_htsig.bool()
    if mask.any():
        loss = loss + F.binary_cross_entropy_with_logits(
                        htsig_l[mask].float(), htsig_gt[mask].float())
    return loss

def bit_acc(logits, targets):
    return ((logits.sigmoid() > 0.5).float() == targets).float().mean().item()

def prot_acc(logits, targets):
    return (logits.argmax(1) == targets).float().mean().item()

def get_scheduler_with_warmup(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1 + torch.cos(torch.tensor(progress * 3.14159265)).item())
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Dataset ────────────────────────────────────────────────────────────────
class ProgDistillDataset(torch.utils.data.Dataset):
    """
    Returns BOTH high-res and low-res for every sample.
    Teacher always gets high-res.
    Student gets progressive mix (interpolated to 8000 for consistent size).
    """
    def __init__(self, data, augment=False):
        self.high, self.low, self.prot, self.lsig, self.htsig, self.has_htsig = data
        self.augment   = augment
        self.low_ratio = 0.0

    def set_low_ratio(self, ratio):
        self.low_ratio = ratio

    def __len__(self):
        return len(self.prot)

    def __getitem__(self, idx):
        high = self.high[idx].clone()
        low  = self.low[idx].clone()

        if self.augment:
            theta = torch.empty(1).uniform_(0, 2 * 3.14159265).item()
            cos_t = torch.cos(torch.tensor(theta))
            sin_t = torch.sin(torch.tensor(theta))
            for iq in (high, low):
                real  = iq[0].clone()
                iq[0] = real * cos_t - iq[1] * sin_t
                iq[1] = real * sin_t + iq[1] * cos_t

        # Student signal — progressive mix, normalized to 8000
        use_low    = torch.rand(1).item() < self.low_ratio
        student_sig = low if use_low else high
        if student_sig.shape[-1] != 8000:
            student_sig = F.interpolate(student_sig.unsqueeze(0), size=8000,
                                        mode='linear', align_corners=False).squeeze(0)

        # Teacher always gets real high-res
        return high, student_sig, self.prot[idx], self.lsig[idx], \
               self.htsig[idx], self.has_htsig[idx]


def get_progressive_ratio(epoch, total_epochs=40):
    phase = (epoch - 1) / (total_epochs - 1)
    if phase < 0.2:    return 0.0
    elif phase < 0.4:  return 0.25
    elif phase < 0.6:  return 0.5
    elif phase < 0.8:  return 0.75
    else:              return 1.0


# ── Shared Model Architecture ──────────────────────────────────────────────
class IQEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(2, 64,  kernel_size=7, stride=2, padding=3), nn.BatchNorm1d(64),  nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2), nn.BatchNorm1d(128), nn.GELU(),
            nn.Conv1d(128, 256, kernel_size=3, stride=2, padding=1), nn.BatchNorm1d(256), nn.GELU(),
            nn.Conv1d(256, 384, kernel_size=3, stride=2, padding=1), nn.BatchNorm1d(384), nn.GELU(),
            nn.Conv1d(384, embed_dim, kernel_size=3, stride=2, padding=1), nn.BatchNorm1d(embed_dim), nn.GELU(),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dim_feedforward=2048,
            dropout=0.1, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6,
                                                  enable_nested_tensor=False)
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        feat   = self.cnn(x)
        feat_t = self.transformer(feat.permute(0,2,1)).permute(0,2,1)
        return self.pool(feat_t).squeeze(-1)


class WiFiDecoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.prot_head  = nn.Sequential(nn.Linear(embed_dim, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, NUM_PROT))
        self.lsig_head  = nn.Sequential(nn.Linear(embed_dim, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, LSIG_LEN))
        self.htsig_head = nn.Sequential(nn.Linear(embed_dim, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, HTSIG_LEN))

    def forward(self, emb):
        return self.prot_head(emb), self.lsig_head(emb), self.htsig_head(emb)


class WiFiModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = IQEncoder()
        self.decoder = WiFiDecoder()

    def forward(self, x):
        emb = self.encoder(x)
        prot_l, lsig_l, htsig_l = self.decoder(emb)
        return prot_l, lsig_l, htsig_l, emb, emb, emb


# ── Evaluate (always on low-res) ───────────────────────────────────────────
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, data, augment=False):
        self.high, self.low, self.prot, self.lsig, self.htsig, self.has_htsig = data
        self.augment = augment

    def __len__(self):
        return len(self.prot)

    def __getitem__(self, idx):
        high = self.high[idx].clone()
        low  = self.low[idx].clone()
        if self.augment:
            theta = torch.empty(1).uniform_(0, 2 * 3.14159265).item()
            cos_t = torch.cos(torch.tensor(theta))
            sin_t = torch.sin(torch.tensor(theta))
            for iq in (high, low):
                real  = iq[0].clone()
                iq[0] = real * cos_t - iq[1] * sin_t
                iq[1] = real * sin_t + iq[1] * cos_t
        return high, low, self.prot[idx], self.lsig[idx], self.htsig[idx], self.has_htsig[idx]

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_loss = tot_prot = tot_lsig = 0
    for high_iq, low_iq, prot, lsig, htsig, has_htsig in loader:
        iq        = low_iq.to(DEVICE, non_blocking=True)
        prot      = prot.to(DEVICE, non_blocking=True)
        lsig      = lsig.to(DEVICE, non_blocking=True)
        htsig     = htsig.to(DEVICE, non_blocking=True)
        has_htsig = has_htsig.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            prot_l, lsig_l, htsig_l, _, _, _ = model(iq)
        loss = task_loss_balanced(
            prot_l.float(), lsig_l.float(), htsig_l.float(),
            prot, lsig.float(), htsig.float(), has_htsig)
        tot_loss += loss.item()
        tot_prot += prot_acc(prot_l, prot)
        tot_lsig += bit_acc(lsig_l, lsig)
    n = len(loader)
    return tot_loss/n, tot_prot/n, tot_lsig/n


# ── Phase 1: Train Teacher on High-Res ────────────────────────────────────
print("\n" + "="*60)
print("PHASE 1: Training Teacher on High-Res Signals")
print("="*60)

TEACHER_EPOCHS = 40
TEACHER_LR     = 3e-4
WARMUP_EPOCHS  = 3

# Teacher dataset — always high-res
class HighResDataset(torch.utils.data.Dataset):
    def __init__(self, data, augment=False):
        self.high, self.low, self.prot, self.lsig, self.htsig, self.has_htsig = data
        self.augment = augment

    def __len__(self):
        return len(self.prot)

    def __getitem__(self, idx):
        high = self.high[idx].clone()
        low  = self.low[idx].clone()
        if self.augment:
            theta = torch.empty(1).uniform_(0, 2 * 3.14159265).item()
            cos_t = torch.cos(torch.tensor(theta))
            sin_t = torch.sin(torch.tensor(theta))
            for iq in (high, low):
                real  = iq[0].clone()
                iq[0] = real * cos_t - iq[1] * sin_t
                iq[1] = real * sin_t + iq[1] * cos_t
        return high, low, self.prot[idx], self.lsig[idx], self.htsig[idx], self.has_htsig[idx]

teacher_train_ds = HighResDataset(train_data, augment=True)
test_ds          = AugmentedDataset(test_data, augment=False)
teacher_loader   = DataLoader(teacher_train_ds, batch_size=256, shuffle=True,
                               num_workers=0, pin_memory=True)
test_loader      = DataLoader(test_ds, batch_size=256, shuffle=False,
                               num_workers=0, pin_memory=True)

teacher_model  = WiFiModel().to(DEVICE)
t_opt          = torch.optim.AdamW(teacher_model.parameters(), lr=TEACHER_LR, weight_decay=1e-4)
t_sched        = get_scheduler_with_warmup(t_opt, WARMUP_EPOCHS, TEACHER_EPOCHS)
t_scaler       = torch.amp.GradScaler('cuda')
best_t_prot    = 0.0

total_params = sum(p.numel() for p in teacher_model.parameters())
print(f"Teacher parameters: {total_params:,} ({total_params*4/1e6:.1f} MB)")
print(f"\n{'Ep':>3} {'tr_loss':>8} {'tr_prot':>8} {'va_prot':>8} {'va_lsig':>8} {'lr':>8} {'time':>6}")
print("-" * 62)

for epoch in range(1, TEACHER_EPOCHS + 1):
    teacher_model.train()
    t0 = time.time()
    tot_loss = tot_prot = tot_lsig = 0
    t_opt.zero_grad()

    for i, (high_iq, low_iq, prot, lsig, htsig, has_htsig) in enumerate(teacher_loader):
        # Teacher trains on HIGH-RES
        high_iq   = high_iq.to(DEVICE, non_blocking=True)
        prot      = prot.to(DEVICE, non_blocking=True)
        lsig      = lsig.to(DEVICE, non_blocking=True)
        htsig     = htsig.to(DEVICE, non_blocking=True)
        has_htsig = has_htsig.to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda'):
            prot_l, lsig_l, htsig_l, _, _, _ = teacher_model(high_iq)
            loss = task_loss_balanced(prot_l, lsig_l, htsig_l,
                                      prot, lsig, htsig, has_htsig)

        t_scaler.scale(loss).backward()
        t_scaler.unscale_(t_opt)
        torch.nn.utils.clip_grad_norm_(teacher_model.parameters(), 1.0)
        t_scaler.step(t_opt)
        t_scaler.update()
        t_opt.zero_grad()

        tot_loss += loss.item()
        tot_prot += prot_acc(prot_l, prot)
        tot_lsig += bit_acc(lsig_l, lsig)

    t_sched.step()
    n  = len(teacher_loader)

    # Evaluate teacher on HIGH-RES
    teacher_model.eval()
    tot_va_prot = tot_va_lsig = 0
    with torch.no_grad():
        for high_iq, low_iq, prot, lsig, htsig, has_htsig in test_loader:
            high_iq   = high_iq.to(DEVICE)
            prot      = prot.to(DEVICE)
            lsig      = lsig.to(DEVICE)
            with torch.amp.autocast('cuda'):
                prot_l, lsig_l, _, _, _, _ = teacher_model(high_iq)
            tot_va_prot += prot_acc(prot_l, prot)
            tot_va_lsig += bit_acc(lsig_l, lsig)

    va_prot = tot_va_prot / len(test_loader)
    va_lsig = tot_va_lsig / len(test_loader)
    lr      = t_opt.param_groups[0]['lr']
    elapsed = time.time() - t0

    if va_prot > best_t_prot:
        best_t_prot = va_prot
        torch.save(teacher_model.state_dict(), '/workspace/prog_distill_teacher.pt')
        marker = ' <- best'
    else:
        marker = ''

    print(f"{epoch:3d} {tot_loss/n:8.4f} {tot_prot/n:8.3f} "
          f"{va_prot:8.3f} {va_lsig:8.3f} {lr:8.2e} {elapsed:5.0f}s{marker}", flush=True)

print(f"\nBest Teacher va_prot (high-res): {best_t_prot:.4f}")

# Load best teacher and freeze it
teacher_model.load_state_dict(torch.load('/workspace/prog_distill_teacher.pt'))
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False
print("Teacher frozen ✅")


# ── Phase 2: Train Student with Progressive Resolution + Online Distillation
print("\n" + "="*60)
print("PHASE 2: Training Student with Progressive Resolution + Online Distillation")
print("Teacher always sees HIGH-RES → provides stable guidance")
print("Student sees progressive resolution mix")
print("="*60)

STUDENT_EPOCHS = 40
STUDENT_LR     = 3e-4
WARMUP_EPOCHS  = 3
ACCUM_STEPS    = 2
ALPHA          = 0.7  # task loss weight

prog_ds     = ProgDistillDataset(train_data, augment=True)
prog_loader = DataLoader(prog_ds, batch_size=256, shuffle=True,
                         num_workers=0, pin_memory=True)

student_model  = WiFiModel().to(DEVICE)
s_opt          = torch.optim.AdamW(student_model.parameters(), lr=STUDENT_LR, weight_decay=1e-4)
s_sched        = get_scheduler_with_warmup(s_opt, WARMUP_EPOCHS, STUDENT_EPOCHS)
s_scaler       = torch.amp.GradScaler('cuda')
best_s_prot    = 0.0

print(f"Student parameters: {sum(p.numel() for p in student_model.parameters()):,}")
print(f"\n{'Ep':>3} {'lo_ratio':>9} {'tr_loss':>8} {'tr_prot':>8} {'dist_l':>8} {'va_prot':>8} {'va_lsig':>8} {'time':>6}")
print("-" * 80)

for epoch in range(1, STUDENT_EPOCHS + 1):
    low_ratio = get_progressive_ratio(epoch, STUDENT_EPOCHS)
    prog_ds.set_low_ratio(low_ratio)

    student_model.train()
    t0 = time.time()
    tot_loss = tot_prot = tot_lsig = tot_dist = 0
    s_opt.zero_grad()

    for i, (high_iq, student_iq, prot, lsig, htsig, has_htsig) in enumerate(prog_loader):
        high_iq    = high_iq.to(DEVICE, non_blocking=True)
        student_iq = student_iq.to(DEVICE, non_blocking=True)
        prot       = prot.to(DEVICE, non_blocking=True)
        lsig       = lsig.to(DEVICE, non_blocking=True)
        htsig      = htsig.to(DEVICE, non_blocking=True)
        has_htsig  = has_htsig.to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda'):
            # Teacher ALWAYS sees real high-res — stable guidance throughout
            with torch.no_grad():
                _, _, _, t_emb, _, _ = teacher_model(high_iq)

            # Student sees progressive resolution
            prot_l, lsig_l, htsig_l, s_emb, _, _ = student_model(student_iq)

            # Task loss on student predictions
            task_l = task_loss_balanced(prot_l, lsig_l, htsig_l,
                                        prot, lsig, htsig, has_htsig)

            # Distillation loss — student embedding → teacher embedding
            dist_l = (1 - F.cosine_similarity(s_emb, t_emb.detach())).mean()

            # Combined loss
            loss = (ALPHA * task_l + (1 - ALPHA) * dist_l) / ACCUM_STEPS

        s_scaler.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(prog_loader):
            s_scaler.unscale_(s_opt)
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
            s_scaler.step(s_opt)
            s_scaler.update()
            s_opt.zero_grad()

        tot_loss += task_l.item()
        tot_prot += prot_acc(prot_l, prot)
        tot_lsig += bit_acc(lsig_l, lsig)
        tot_dist += dist_l.item()

    s_sched.step()
    n  = len(prog_loader)
    va = evaluate(student_model, test_loader)
    elapsed = time.time() - t0

    if va[1] > best_s_prot:
        best_s_prot = va[1]
        torch.save(student_model.state_dict(), '/workspace/prog_distill_student_best.pt')
        marker = ' <- best'
    else:
        marker = ''

    print(f"{epoch:3d} {low_ratio:9.2f} {tot_loss/n:8.4f} {tot_prot/n:8.3f} "
          f"{tot_dist/n:8.4f} {va[1]:8.3f} {va[2]:8.3f} {elapsed:5.0f}s{marker}", flush=True)

print(f"\nBest Student va_prot (low-res eval): {best_s_prot:.4f}")

# ── Per-class accuracy ─────────────────────────────────────────────────────
student_model.load_state_dict(torch.load('/workspace/prog_distill_student_best.pt'))
student_model.eval()
correct = torch.zeros(4)
total   = torch.zeros(4)

with torch.no_grad():
    for high_iq, low_iq, prot, *_ in test_loader:
        iq     = low_iq.to(DEVICE)
        prot_g = prot.to(DEVICE)
        with torch.amp.autocast('cuda'):
            prot_l, _, _, _, _, _ = student_model(iq)
        preds = prot_l.argmax(1)
        for c in range(4):
            mask = (prot_g == c)
            correct[c] += (preds[mask] == prot_g[mask]).sum().cpu()
            total[c]   += mask.sum().cpu()

print("\nPer-class accuracy (Proper Progressive + Distillation):")
print(f"{'Class':>8} {'Prog+Dist':>10} {'CNN+Trans':>12} {'Diff':>8}")
print("-" * 44)
cnn_trans = [97.0, 91.9, 58.0, 50.0]
for c in range(4):
    acc  = correct[c]/(total[c]+1e-8) * 100
    diff = acc.item() - cnn_trans[c]
    print(f"{PROT_NAMES[c]:>8}: {acc.item():>9.1f}%  {cnn_trans[c]:>10.1f}%  {diff:>+6.1f}%")

Device: cuda
GPU: NVIDIA GeForce RTX 5090
Loading data...
Train: 86,437 | Test: 9,725

PHASE 1: Training Teacher on High-Res Signals
Teacher parameters: 20,356,748 (81.4 MB)

 Ep  tr_loss  tr_prot  va_prot  va_lsig       lr   time
--------------------------------------------------------------
  1   1.1256    0.822    0.944    0.918 2.00e-04    50s <- best
  2   0.6709    0.951    0.977    0.935 3.00e-04    50s <- best
  3   0.5379    0.971    0.978    0.952 3.00e-04    50s <- best
  4   0.4232    0.979    0.983    0.962 2.99e-04    50s <- best
  5   0.3439    0.986    0.990    0.972 2.98e-04    50s <- best
  6   0.2714    0.991    0.991    0.977 2.95e-04    50s <- best
  7   0.2419    0.992    0.992    0.981 2.91e-04    50s <- best
  8   0.2150    0.993    0.994    0.983 2.87e-04    50s <- best
  9   0.1972    0.993    0.991    0.983 2.81e-04    50s
 10   0.1872    0.993    0.994    0.985 2.74e-04    50s
 11   0.1718    0.994    0.995    0.986 2.67e-04    50s <- best
 12   0.1626    0.